## ML_1027: Implement Dropout

**Difficulty**: Easy | **TorchCode**: #17

### Problem
Implement inverted Dropout as an `nn.Module`. In training, randomly zero elements with probability `p` and scale survivors by `1/(1-p)`. In eval mode, return the input unchanged.

### Formula
$$\text{Dropout}(x_i) = \begin{cases} 0 & \text{with prob } p \\ \dfrac{x_i}{1-p} & \text{with prob } 1-p \end{cases} \quad (\text{training})$$

In [ ]:
import torch
import torch.nn as nn

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        mask = (torch.rand_like(x) > self.p).float()
        return x * mask / (1 - self.p)

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Eval mode is identity ---
d = MyDropout(p=0.5)
assert isinstance(d, nn.Module)
d.eval()
x = torch.randn(4, 8)
assert torch.equal(d(x), x)

# --- Test 2: Training — zeros and scaling ---
torch.manual_seed(42)
d.train()
x = torch.ones(1000)
out = d(x)
assert (out == 0).any()
non_zero = out[out != 0]
assert torch.allclose(non_zero, torch.full_like(non_zero, 2.0), atol=1e-5)

# --- Test 3: Drop rate ≈ p ---
torch.manual_seed(0)
d2 = MyDropout(p=0.3)
d2.train()
out = d2(torch.ones(10000))
frac = (out == 0).float().mean().item()
assert 0.25 < frac < 0.35

# --- Test 4: Gradient flow ---
d.train()
x = torch.randn(4, 8, requires_grad=True)
d(x).sum().backward()
assert x.grad is not None

print('All tests passed!')